## 1. Install Dependencies and Import Libraries

Install and import necessary libraries including PyTorch, NumPy, Matplotlib, and scikit-learn for data preprocessing and visualization.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Install required packages
import subprocess
import sys

packages = ['numpy', 'matplotlib', 'scikit-learn', 'tensorflow']
print("Installing required packages...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ {package} installation had issues, attempting without quiet mode...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        except Exception as e2:
            print(f"✗ {package} installation failed: {e2}")
print("\nPackage installation completed!")

Installing required packages...


CalledProcessError: Command '['e:\\BASIC ENCODER\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'tensorflow']' returned non-zero exit status 1.

## 2. Load and Prepare Dataset

Load a simple dataset (MNIST), normalize the data to the range [0, 1], and split into training and testing sets.

In [ ]:
# Load MNIST dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Flatten and normalize data to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten the images
x_train_flat = x_train.reshape(-1, 28 * 28)
x_test_flat = x_test.reshape(-1, 28 * 28)

# Display dataset information
print(f"Training set shape: {x_train_flat.shape}")
print(f"Testing set shape: {x_test_flat.shape}")
print(f"Data range: [{x_train_flat.min():.3f}, {x_train_flat.max():.3f}]")
print(f"Training set mean: {x_train_flat.mean():.3f}, std: {x_train_flat.std():.3f}")

## 3. Define the Autoencoder Architecture

Build an autoencoder model with:
- **Encoder**: Compresses input data (784 → 128 → 64 → 32) into a latent representation
- **Decoder**: Reconstructs the original input from the latent code (32 → 64 → 128 → 784)

In [ ]:
# Define input dimension
input_dim = 28 * 28  # 784
latent_dim = 32

# Build the encoder
input_img = layers.Input(shape=(input_dim,))
encoded = layers.Dense(128, activation='relu')(input_img)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(32, activation='relu')(encoded)
latent = layers.Dense(latent_dim, activation='relu')(encoded)

# Build the decoder
decoded = layers.Dense(32, activation='relu')(latent)
decoded = layers.Dense(64, activation='relu')(decoded)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)

# Create the autoencoder model
autoencoder = Model(input_img, decoded)

# Display model architecture
print("Autoencoder Architecture:")
print("=" * 50)
print(f"Input dimension: {input_dim}")
print(f"Latent dimension: {latent_dim}")
print("=" * 50)
autoencoder.summary()

## 4. Compile the Model

Configure the autoencoder with an appropriate optimizer (Adam) and a loss function (Mean Squared Error) suitable for reconstruction tasks.

In [ ]:
# Compile the autoencoder model
optimizer = Adam(learning_rate=0.001)
autoencoder.compile(
    optimizer=optimizer,
    loss='mse',  # Mean Squared Error
    metrics=['mae']  # Mean Absolute Error as additional metric
)

print("Model compiled successfully!")
print(f"Optimizer: Adam (learning_rate=0.001)")
print(f"Loss function: Mean Squared Error (MSE)")
print(f"Metrics: Mean Absolute Error (MAE)")

## 5. Train the Autoencoder

Train the autoencoder on the training dataset for multiple epochs, monitoring training and validation loss to ensure proper convergence.

In [ ]:
# Train the autoencoder
print("Training the autoencoder...")
print("=" * 60)

history = autoencoder.fit(
    x_train_flat, x_train_flat,
    epochs=30,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=1
)

print("=" * 60)
print("Training completed!")

## 6. Evaluate Reconstruction Error

Calculate and report reconstruction error metrics on the test set, including Mean Squared Error (MSE), Mean Absolute Error (MAE), and other relevant performance metrics.

In [ ]:
# Evaluate on test set
print("Evaluating reconstruction performance on test set...")
print("=" * 60)

# Get predictions on test set
x_test_pred = autoencoder.predict(x_test_flat)

# Calculate reconstruction errors
mse = mean_squared_error(x_test_flat, x_test_pred)
mae = mean_absolute_error(x_test_flat, x_test_pred)
rmse = np.sqrt(mse)

# Calculate per-sample reconstruction errors
per_sample_mse = np.mean((x_test_flat - x_test_pred) ** 2, axis=1)

# Print metrics
print(f"\nReconstruction Error Metrics (Test Set):")
print(f"  Mean Squared Error (MSE):     {mse:.6f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"  Mean Absolute Error (MAE):    {mae:.6f}")
print(f"\nPer-Sample MSE Statistics:")
print(f"  Mean:   {per_sample_mse.mean():.6f}")
print(f"  Std:    {per_sample_mse.std():.6f}")
print(f"  Min:    {per_sample_mse.min():.6f}")
print(f"  Max:    {per_sample_mse.max():.6f}")
print(f"  Median: {np.median(per_sample_mse):.6f}")

# Evaluate using model.evaluate()
test_loss, test_mae = autoencoder.evaluate(x_test_flat, x_test_flat, verbose=0)
print(f"\nModel Evaluation Results:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE:        {test_mae:.6f}")
print("=" * 60)

## 7. Visualize Original vs Reconstructed Data

Display side-by-side comparisons of original and reconstructed images to qualitatively assess the autoencoder's performance.

In [ ]:
# Visualize original vs reconstructed images
n_images = 10
plt.figure(figsize=(20, 4))

for i in range(n_images):
    # Original images
    ax = plt.subplot(2, n_images, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.title(f"Original {i+1}")
    plt.axis('off')
    
    # Reconstructed images
    ax = plt.subplot(2, n_images, i + 1 + n_images)
    reconstructed = x_test_pred[i].reshape(28, 28)
    plt.imshow(reconstructed, cmap='gray')
    plt.title(f"Reconstructed {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.suptitle("Original vs Reconstructed Images", fontsize=16, y=1.02)
plt.show()

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# MSE Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE Loss', fontsize=12)
axes[0].set_title('Model Loss over Epochs', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# MAE Metric
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MAE', fontsize=12)
axes[1].set_title('Model MAE over Epochs', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize reconstruction error distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Histogram of per-sample reconstruction errors
axes[0].hist(per_sample_mse, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(per_sample_mse.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {per_sample_mse.mean():.6f}')
axes[0].axvline(np.median(per_sample_mse), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(per_sample_mse):.6f}')
axes[0].set_xlabel('Per-Sample MSE', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Reconstruction Errors', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Sorted reconstruction errors
sorted_errors = np.sort(per_sample_mse)
axes[1].plot(sorted_errors, linewidth=2, color='steelblue')
axes[1].fill_between(range(len(sorted_errors)), sorted_errors, alpha=0.3, color='steelblue')
axes[1].set_xlabel('Sample Index (sorted by error)', fontsize=12)
axes[1].set_ylabel('MSE', fontsize=12)
axes[1].set_title('Sorted Reconstruction Errors', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize best and worst reconstructions
best_indices = np.argsort(per_sample_mse)[:5]  # 5 best
worst_indices = np.argsort(per_sample_mse)[-5:]  # 5 worst

fig, axes = plt.subplots(4, 5, figsize=(15, 10))

# Best reconstructions
for idx, sample_idx in enumerate(best_indices):
    ax = axes[0, idx]
    ax.imshow(x_test[sample_idx], cmap='gray')
    ax.set_title(f"Best {idx+1}\nMSE: {per_sample_mse[sample_idx]:.4f}", fontsize=10)
    ax.axis('off')
    
    ax = axes[1, idx]
    ax.imshow(x_test_pred[sample_idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"Reconstructed", fontsize=10)
    ax.axis('off')

# Worst reconstructions
for idx, sample_idx in enumerate(worst_indices):
    ax = axes[2, idx]
    ax.imshow(x_test[sample_idx], cmap='gray')
    ax.set_title(f"Worst {idx+1}\nMSE: {per_sample_mse[sample_idx]:.4f}", fontsize=10)
    ax.axis('off')
    
    ax = axes[3, idx]
    ax.imshow(x_test_pred[sample_idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"Reconstructed", fontsize=10)
    ax.axis('off')

fig.suptitle('Best vs Worst Reconstructions', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Basic Autoencoder for Data Reconstruction

This notebook implements a basic autoencoder neural network to reconstruct input data. We'll train it on a simple dataset and evaluate its reconstruction error.